<a href="https://colab.research.google.com/github/ext-colorful/LangChain-Essentials/blob/main/%F0%9F%A7%B1Building_Blocks%F0%9F%A7%B1L5_tools_with_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[课程地址](https://academy.langchain.com/courses/take/langchain-essentials-python/lessons/69388326-lesson-5-tools-with-mcp)😁
[源码地址](https://github.com/langchain-ai/lca-langchainV1-essentials/blob/main/python/L1_fast_agent.ipynb)😁
[LANGSMITH官网](https://smith.langchain.com)😁
[LANGCHAIN智能助手](https://chat.langchain.com)😁
[免费的智能体代理商](https://api.chatanywhere.tech)

In Lessons 2–7, you will learn how to use some of the fundamental building blocks in LangChain. These lessons explain and complement create_agent, and you’ll find them useful when creating your own agents. Each lesson is concise and focused.
> 在课程2-7中，你将学习如何使用LangChain中的一些基本构建模块。这些课程解释并补充了create_agent，当你创建自己的代理时，你会发现它们很有用。每个课程都简洁而专注。

## Learn to use the LangChain MCP adapter to access the world of MCP tools.
> 学习使用LangChain MCP适配器来访问MCP工具的世界。

# Tools with MCP ⏰
The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `langchain-mcp-adapters`.
> 模型上下文协议 (MCP) 提供了一种标准化的方式，将 AI 代理连接到外部工具和数据源。让我们使用 langchain-mcp-adapters. 连接到 MCP 服务器。

## Setup
Load and/or check for needed environmental variables
> 加载和/或检查所需的环境变量

In [1]:
!pip install -q langgraph==1.0.3 langchain==1.0.8 langchain-openai==1.0.3 langchain-community==0.4.1 langgraph-cli[inmem]==0.4.7 langchain-mcp-adapters==0.1.13
# Used to securely store your API key
# 用于安全存储您的API密钥
from google.colab import userdata
import os

# Retrieve the API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_BASE"] = userdata.get('OPENAI_API_BASE')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain_agent_L5_tools_with_mcp"

### ⚠️执行下边的代码报这个错误
--------------------------------------------------------------------------- UnsupportedOperation Traceback (most recent call last) /tmp/ipython-input-2155988407.py in <cell line: 1>() 18 19 # Load tools from the MCP server ---> 20 mcp_tools = await mcp_client.get_tools() 21 print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}") 19 frames /usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py in fileno(self) 309 return self._original_stdstream_copy 310 else: --> 311 raise io.UnsupportedOperation("fileno") 312 313 def _watch_pipe_fd(self): UnsupportedOperation: fileno

这类报错的味道，一 sniff 就能闻出是 Jupyter / IPython 环境 自带的老毛病 —— 某些库尝试去获取标准输出的 文件描述符（file descriptor），结果发现它根本不存在，于是就扔出：

UnsupportedOperation: fileno

画面大概像这样：
“我要拿到 stdout 的文件句柄！”
Jupyter：我没有实体文件流啊哥……
Python：那我只能报错了。

原因：

ipykernel.iostream.OutStream（也就是 notebook 里替代 sys.stdout 的家伙）不支持 .fileno() 方法。
MCP Client 或底层库在尝试用某种方式 监听 / 轮询标准输入输出，需要 .fileno()，但 notebook 下没这个功能，于是炸了。

通常发生在：

用 asyncio + Jupyter

某个库要阻塞式读取 stdin/stdout

MCP client 里可能用了 asyncio.get_event_loop().add_reader(...)，它需要文件描述符

在 Notebook 环境就直接跪了

### 解决方法：在 Notebook 环境中 patch 掉 fileno（不推荐但可救急）
```python
import ipykernel.iostream
ipykernel.iostream.OutStream.fileno = lambda self: 1
```

这基本是“虚构一个文件描述符”，有风险，但有时能骗过库完成初始化。

In [2]:
import ipykernel.iostream
ipykernel.iostream.OutStream.fileno = lambda self: 1

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio

nest_asyncio.apply()

# Connect to the mcp-time server for timezone-aware operations
# This Go-based server provides tools for current time, relative time parsing,
# timezone conversion, duration arithmetic, and time comparison
mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Loaded 5 MCP tools: ['add_time', 'compare_time', 'convert_timezone', 'current_time', 'relative_time']


Create an agent with the MCP-provided time tools.
> 使用MCP提供的时间工具创建一个代理。

In [4]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="openai:gpt-3.5-turbo",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

Ask about the current time in San Francisco.
> 询问旧金山的当前时间。

In [5]:
result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What's the time in SF right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


================================ Human Message =================================

What's the time in SF right now?
================================== Ai Message ==================================
Tool Calls:
  current_time (call_Ff3L451tgGhW6nw7PTAPWXuW)
 Call ID: call_Ff3L451tgGhW6nw7PTAPWXuW
  Args:
    timezone: America/Los_Angeles
================================= Tool Message =================================
Name: current_time

2025-11-22T07:29:17-08:00
================================== Ai Message ==================================

The current time in San Francisco is approximately 7:29 AM on November 22, 2025.
